# 03 - Neighborhood "feel" features (quiet, green, access, form, centrality)

Replaces the shop-mix similarity from notebook 02 with features that match what I actually care about:
quiet streets, green space, access to a library/park/groceries at bike and drive scale, moderate density,
and moderate distance to downtown. **No income and no boutique/shop-mix features.** Price will be a budget filter later, not a similarity feature.

Inputs: the parquet cache from notebook 02 (POIs, walk-network nodes/edges, boundaries) plus two new extracts from the same Geofabrik PBF (road classes, park polygons). `data/labels.csv` holds the labeled places.

Test before trusting anything (write the answer down first): does the home-area anchor's nearest-Houston list include West U and Museum District? Do Midtown and Richmond-like places look different?

Similarity here is **weighted Euclidean distance on z-scored features**, not cosine: absolute level matters (a quiet street is not the same as "a bit less busy than average").

In [ ]:
# Kaggle/Colab: uncomment. Locally: pip install in your venv instead.
# %pip install -q geopandas pyarrow pyrosm "shapely>=2" scipy

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, shapely
from shapely import STRtree
from scipy.spatial import cKDTree
from pyproj import Transformer
from pyrosm import OSM
warnings.filterwarnings("ignore")

CACHE = Path("/kaggle/working/cache") if os.path.exists("/kaggle") else Path("../data/interim")
PBF_DIR = CACHE / "pbf"
STATE = {"houston": "texas", "austin": "texas", "seattle": "washington", "atlanta": "georgia"}
CITIES = ["houston"]   # add "austin", "seattle", "atlanta" once Houston looks right
# Approximate downtown points (lat, lon) - VERIFY ON A MAP
DOWNTOWN = {"houston": (29.7604, -95.3698), "austin": (30.2672, -97.7431),
            "seattle": (47.6062, -122.3321), "atlanta": (33.7490, -84.3880)}
SPACING_M, RADIUS_M, NMS_M = 400, 800, 1600
BIG_PARK_HA = 10          # "large park" threshold (Hermann Park is far above this)
MIN_POI_800 = 10          # drop empty land: cell needs this many POIs of any kind within 800 m

def city_crs(bbox):
    lon = (bbox[0] + bbox[2]) / 2
    return f"EPSG:{32600 + int((lon + 180) // 6) + 1}"   # must match notebook 02

def latest(pattern):
    fs = sorted(CACHE.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not fs: raise FileNotFoundError(f"{pattern} not in {CACHE}; run notebook 02 first")
    return fs[-1]

def pbf_path(city): return PBF_DIR / f"{STATE[city]}-latest.osm.pbf"

# labels: ../data/labels.csv locally; on Kaggle upload it to /kaggle/working or attach it as a dataset
cands = [Path("../data/labels.csv"), Path("/kaggle/working/labels.csv")] + list(Path("/kaggle/input").glob("**/labels.csv"))
LABELS = next((p for p in cands if p.exists()), None)
assert LABELS is not None, "labels.csv not found"
labels = pd.read_csv(LABELS).dropna(subset=["lat", "lon"]).reset_index(drop=True)
print(labels[["name", "city", "label"]].to_string(index=False))

## 1. Extra extracts from the PBF (cached): road classes and park polygons

In [ ]:
MAJOR = {"motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "secondary", "secondary_link"}
FREEWAY = {"motorway", "motorway_link", "trunk", "trunk_link"}

def load_roads(city, bbox, crs):
    fr, fm = CACHE / f"{city}_roads_class.parquet", CACHE / f"{city}_major_pts.parquet"
    if fr.exists() and fm.exists():
        return pd.read_parquet(fr), pd.read_parquet(fm)
    ed = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_network(network_type="driving").to_crs(crs)
    ed = ed[ed.geometry.notna()].reset_index(drop=True)
    hw = ed["highway"].astype(str)
    mid = ed.geometry.interpolate(0.5, normalized=True)
    roads = pd.DataFrame({"x": mid.x.values, "y": mid.y.values, "len": ed.geometry.length.values,
                          "major": hw.isin(MAJOR).values, "freeway": hw.isin(FREEWAY).values})
    rows = []   # points every ~50 m along major roads, for nearest-distance queries
    for geom, fw in zip(ed.geometry[roads["major"]], roads["freeway"][roads["major"]]):
        for t in np.linspace(0, 1, max(int(geom.length // 50), 1) + 1):
            p = geom.interpolate(t, normalized=True); rows.append((p.x, p.y, bool(fw)))
    pts = pd.DataFrame(rows, columns=["x", "y", "freeway"])
    roads.to_parquet(fr); pts.to_parquet(fm)
    return roads, pts

def load_parks(city, bbox, crs):
    f = CACHE / f"{city}_parks.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    g = OSM(str(pbf_path(city)), bounding_box=list(bbox)).get_data_by_custom_criteria(
        custom_filter={"leisure": ["park", "garden", "nature_reserve"]}, filter_type="keep",
        keep_nodes=False, keep_ways=True, keep_relations=True)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(crs)
    g = g[["geometry"]].reset_index(drop=True)
    g["area_ha"] = g.geometry.area / 1e4
    g.to_parquet(f)
    return g

def tree(xy): return cKDTree(xy) if len(xy) else None

D = {}
for c in CITIES:
    poly = gpd.read_file(CACHE / f"{c}_boundary.geojson").geometry.iloc[0]
    crs = city_crs(poly.bounds)
    poi = pd.read_parquet(latest(f"{c}_pois_pbf_*.parquet"))
    nodes = pd.read_parquet(latest(f"{c}_nodes_pbf_*.parquet"))
    edges = pd.read_parquet(latest(f"{c}_edges_pbf_*.parquet"))
    roads, mpts = load_roads(c, poly.bounds, crs)
    parks = load_parks(c, poly.bounds, crs)
    big = parks[parks["area_ha"] >= BIG_PARK_HA]
    def pts(tags, p=poi):
        q = p[p["tag"].isin(tags)]; return tree(q[["x", "y"]].values)
    D[c] = dict(
        poly=poly, crs=crs,
        poi_all=tree(poi[["x", "y"]].values),
        library=pts(["amenity=library"]), market=pts(["shop=supermarket"]), hospital=pts(["amenity=hospital"]),
        school=pts(["amenity=school"]), eat=pts(["amenity=restaurant", "amenity=cafe", "amenity=fast_food"]),
        ntree=cKDTree(nodes[["x", "y"]].values), ndeg=nodes["deg"].values,
        etree=cKDTree(edges[["x", "y"]].values), elen=edges["len"].values,
        rtree=cKDTree(roads[["x", "y"]].values), rlen=roads["len"].values, rmaj=roads["major"].values,
        majtree=tree(mpts[["x", "y"]].values), fwtree=tree(mpts.loc[mpts["freeway"], ["x", "y"]].values),
        pgeom=parks.geometry.values, ptree=STRtree(parks.geometry.values),
        btree=STRtree(big.geometry.values) if len(big) else None)
    print(c, "| POIs", len(poi), "| roads", len(roads), "| parks", len(parks), f"({len(big)} >= {BIG_PARK_HA} ha)")

## 2. Features per point (works for labeled places and for grid cells)

In [ ]:
GROUPS = {
    "quiet":    ["major_road_share", "dist_major", "dist_freeway"],
    "green":    ["park_frac", "dist_park", "dist_bigpark"],
    "vicinity": ["dist_library", "n_library_3km", "n_market_3km", "n_hospital_5km", "n_school_1500", "n_eat_1500"],
    "form":     ["intersection_density", "mean_edge_len", "log_poi_800"],
    "center":   ["dist_downtown_km"],
}
FEATS = [f for cols in GROUPS.values() for f in cols]
# Tune these per anchor. Walkability-type "form" features get low weight on purpose.
WEIGHTS = {"quiet": 1.0, "green": 1.0, "vicinity": 1.0, "form": 0.5, "center": 0.5}
W = np.array([WEIGHTS[g] / np.sqrt(len(cols)) for g, cols in GROUPS.items() for _ in cols])
LOGCOLS = [f for f in FEATS if f.startswith(("dist_", "n_")) or f in ("intersection_density", "mean_edge_len")]

def to_xy(city, lat, lon):
    return Transformer.from_crs("EPSG:4326", D[city]["crs"], always_xy=True).transform(lon, lat)

def to_latlon(city, x, y):
    lon, lat = Transformer.from_crs(D[city]["crs"], "EPSG:4326", always_xy=True).transform(x, y)
    return lat, lon

def near(t, pts, cap):
    if t is None: return np.full(len(pts), float(cap))
    return np.minimum(t.query(np.c_[shapely.get_x(pts), shapely.get_y(pts)])[0], cap)

def pdist(t, pts, cap):
    if t is None: return np.full(len(pts), float(cap))
    _, dist = t.query_nearest(pts, return_distance=True, all_matches=False)
    return np.minimum(dist, cap)

def count(t, P, r):
    return np.zeros(len(P)) if t is None else t.query_ball_point(P, r, return_length=True).astype(float)

def compute(c, P):
    d = D[c]; n = len(P); area_km2 = np.pi * (RADIUS_M / 1000) ** 2
    pts = shapely.points(P[:, 0], P[:, 1])
    # quiet: share of nearby road length that is major, distance to major/freeway
    tot, maj = np.zeros(n), np.zeros(n)
    for k, idx in enumerate(d["rtree"].query_ball_point(P, RADIUS_M)):
        if idx:
            l = d["rlen"][idx]; tot[k] = l.sum(); maj[k] = l[d["rmaj"][idx]].sum()
    # green: park area share within 800 m
    bufs = shapely.buffer(pts, RADIUS_M); pf = np.zeros(n)
    for k, b in enumerate(bufs):
        ids = d["ptree"].query(b)
        if len(ids): pf[k] = shapely.area(shapely.intersection(d["pgeom"][ids], b)).sum() / b.area
    # form: intersection density, mean edge length
    inter, mel = np.zeros(n), np.zeros(n)
    for k, idx in enumerate(d["ntree"].query_ball_point(P, RADIUS_M)):
        if idx: inter[k] = (d["ndeg"][idx] >= 3).sum() / area_km2
    for k, idx in enumerate(d["etree"].query_ball_point(P, RADIUS_M)):
        if idx: mel[k] = d["elen"][idx].mean()
    # centrality
    tx = Transformer.from_crs("EPSG:4326", d["crs"], always_xy=True).transform(DOWNTOWN[c][1], DOWNTOWN[c][0])
    out = {
        "major_road_share": np.where(tot > 0, maj / np.maximum(tot, 1), 0.0),
        "dist_major": near(d["majtree"], pts, 5000), "dist_freeway": near(d["fwtree"], pts, 8000),
        "park_frac": pf, "dist_park": pdist(d["ptree"], pts, 3000), "dist_bigpark": pdist(d["btree"], pts, 8000),
        "dist_library": near(d["library"], pts, 8000), "n_library_3km": count(d["library"], P, 3000),
        "n_market_3km": count(d["market"], P, 3000), "n_hospital_5km": count(d["hospital"], P, 5000),
        "n_school_1500": count(d["school"], P, 1500), "n_eat_1500": count(d["eat"], P, 1500),
        "intersection_density": inter, "mean_edge_len": mel,
        "log_poi_800": np.log1p(count(d["poi_all"], P, RADIUS_M)),
        "dist_downtown_km": np.hypot(P[:, 0] - tx[0], P[:, 1] - tx[1]) / 1000,
    }
    return pd.DataFrame(out)[FEATS]

def make_grid(c):
    d = D[c]; w, s, e, n = d["poly"].bounds
    t = Transformer.from_crs("EPSG:4326", d["crs"], always_xy=True)
    (x0, y0), (x1, y1) = t.transform(w, s), t.transform(e, n)
    X, Y = np.meshgrid(np.arange(x0, x1, SPACING_M), np.arange(y0, y1, SPACING_M))
    poly_xy = gpd.GeoSeries([d["poly"]], crs=4326).to_crs(d["crs"]).iloc[0]
    m = shapely.contains_xy(poly_xy, X.ravel(), Y.ravel())
    return np.c_[X.ravel(), Y.ravel()][m]

GRID = {}
for c in CITIES:
    P = make_grid(c); F = compute(c, P)
    GRID[c] = dict(P=P, F=F, ok=(F["log_poi_800"] >= np.log1p(MIN_POI_800)).values)
    print(c, "cells:", len(P), "usable:", int(GRID[c]["ok"].sum()))

## 3. Labeled places: raw values and percentile within their city (usable cells)

In [ ]:
rows, names = [], []
for _, r in labels.iterrows():
    if r["city"] not in D: continue     # e.g. Richmond is outside the Houston boundary
    x, y = to_xy(r["city"], r["lat"], r["lon"])
    rows.append(compute(r["city"], np.array([[x, y]])).iloc[0]); names.append(r["name"])
L = pd.DataFrame(rows, index=names).astype(float)
pd.set_option("display.width", 220, "display.max_columns", 30)
print(L.round(2).T.to_string())

print("\nPercentile of each place among usable cells of its city (0 = lowest, 1 = highest):")
pct = {}
for n_, r in labels.set_index("name").loc[L.index].iterrows():
    G = GRID[r["city"]]["F"][GRID[r["city"]]["ok"]]
    pct[n_] = [(G[f] < L.loc[n_, f]).mean() for f in FEATS]
print(pd.DataFrame(pct, index=FEATS).round(2).to_string())

**Look at this table before the search.** Do the liked places (home area, West U, Museum District) share a pattern on quiet / green / vicinity?
Do Summer St and Midtown differ (more major roads, higher intersection density)? If the labeled places do not separate on these features, searching will not help yet.

## 4. Search: nearest cells to a labeled anchor (one query per anchor, NMS, biggest gaps)

In [ ]:
def transform(F):
    T = F.copy()
    for c in LOGCOLS: T[c] = np.log1p(T[c])
    return T

ALL = pd.concat([GRID[c]["F"][GRID[c]["ok"]] for c in CITIES])
TA = transform(ALL); MU, SD = TA.mean(), TA.std() + 1e-9
def Z(F): return ((transform(F) - MU) / SD).clip(-3, 3).values

def search(anchor, target, k=8):
    a = labels[labels["name"] == anchor].iloc[0]
    za = Z(L.loc[[anchor]])[0]
    T = GRID[target]; ZT = Z(T["F"])
    d = np.sqrt((((ZT - za) * W) ** 2).sum(1)); d[~T["ok"]] = 1e9
    if a["city"] == target:   # exclude the anchor's own neighborhood
        ax, ay = to_xy(a["city"], a["lat"], a["lon"])
        d[np.hypot(T["P"][:, 0] - ax, T["P"][:, 1] - ay) < NMS_M] = 1e9
    chosen = []
    for j in np.argsort(d):
        if d[j] >= 1e9: break
        if all(np.hypot(*(T["P"][j] - T["P"][c])) >= NMS_M for c in chosen): chosen.append(j)
        if len(chosen) == k: break
    out = []
    for j in chosen:
        lat, lon = to_latlon(target, *T["P"][j])
        gap = np.argsort(-np.abs((ZT[j] - za) * W))[:3]
        out.append(dict(dist=round(float(d[j]), 2), lat=round(lat, 5), lon=round(lon, 5),
                        map=f"https://maps.google.com/?q={lat:.5f},{lon:.5f}",
                        biggest_gaps=", ".join(np.array(FEATS)[gap])))
    return pd.DataFrame(out)

pd.set_option("display.max_colwidth", 90)
for anchor in L.index:
    if labels.set_index("name").loc[anchor, "label"] not in ("like", "like_conditional"): continue
    for target in CITIES:
        print(f"\n=== {anchor}  ->  {target} (lower dist = closer) ===")
        print(search(anchor, target).to_string(index=False))

## 5. Not built yet (next steps, in order)
1. **ACS census features** (share of households with children, 65+ share, owner share, median rent) by tract, joined to cells. Rent is a *filter*, the family features go in similarity. Needs a Census API call and tract shapes (TIGER/Line).
2. **Tree canopy** (NLCD/USFS raster) for "green" beyond park polygons.
3. **Negative anchors**: use `dislike` rows (needs the previous apartment's location, and a grid that covers Richmond) to push down cells that look more like a disliked place than a liked one.
4. Add Austin / Seattle / Atlanta to `CITIES`. Compare the within-Houston result first; only then trust cross-city.
5. Per-anchor weights (`WEIGHTS`) from the labeled places, as in the CLAUDE.md few-shot plan.